In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import math
import h5py, glob, os
import pandas as pd

from itertools import product
from ang_res_funcs import *

# Event Counts

The goal of this notebook is to collect and print the event counts as a function of calendar year and by tier. Because the maps are split into 360 degree-based bins for each day, the counts are extracted via cluster submission using:
- `get_counts.py`: calculates and stores the cumulative counts for a given tier and year
- `count_submitter.py`: submits `get_counts` for each tier and each year

### Load map files

In [29]:
data_dir = Path('/data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/icecube/event_counts')
count_files = sorted(data_dir.glob('*.txt'))

filenames = [f.stem for f in count_files]
tiers = sorted(set([f[:5] for f in filenames]))
years = sorted(set([f[-4:] for f in filenames]))

In [30]:
# Calculate the counts for each tier
tier_counts = {}

for tier in tiers:
    tier_files = [f for f in count_files if tier in f.stem]
    count = 0
    for tier_file in tier_files:
        with open(tier_file, 'r') as f:
            count += float(f.readlines()[0])

    tier_counts[tier] = count

In [31]:
total = 0
for tier, count in tier_counts.items():
    print(f'Tier {tier[-1]}:  {count/1e8:.02f} x 10^8')
    total += count

print(f'Total:  {total/1e8:.02f} x 10^8')

Tier 1:  5.57 x 10^8
Tier 2:  3.86 x 10^8
Tier 3:  2.53 x 10^8
Tier 4:  0.29 x 10^8
Total:  12.25 x 10^8


In [32]:
# Set which directory we're pulling from

DIRECT = '/data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/icecube/sim_arrays'

# We will load all the .npy arrays below. Refer to the naming conventions to
# understand the np array names. 

# Choose the reconstruction used for each tier. For 2012 with 4 tiers, 
# ShowerPlane is used for Tier 1 and Laputop for Tiers 2, 3 and 4.
# For 2015, it would be ['lap', 'lap']
reco = ['sp', 'lap', 'lap', 'lap']


"""
Naming convention:

mc-  = Monte Carlo (Simulation)
lap- = Laputop (Reconstructed Simulation)
sp-  = ShowerPlane (Reconstructed Simulation)

-p-   = Proton
-he-  = Helium
-o-   = Oxygen
-fe-  = Iron
If particle is not specified, array is concatenated across all four
particle types.

-en-  = Energy
-zen- = Zenith
-az-  = Azimuth
-w-   = Weighting

-t1   = Tier 1, lowest energy bin
-t2   = Tier 2, second lowest energy bin
-t3   = Tier 3, second highest energy bin
-t4   = Tier 4, highest energy bin
-24   = Data from Tiers 2 through 4, concatenated

"""

'\nNaming convention:\n\nmc-  = Monte Carlo (Simulation)\nlap- = Laputop (Reconstructed Simulation)\nsp-  = ShowerPlane (Reconstructed Simulation)\n\n-p-   = Proton\n-he-  = Helium\n-o-   = Oxygen\n-fe-  = Iron\nIf particle is not specified, array is concatenated across all four\nparticle types.\n\n-en-  = Energy\n-zen- = Zenith\n-az-  = Azimuth\n-w-   = Weighting\n\n-t1   = Tier 1, lowest energy bin\n-t2   = Tier 2, second lowest energy bin\n-t3   = Tier 3, second highest energy bin\n-t4   = Tier 4, highest energy bin\n-24   = Data from Tiers 2 through 4, concatenated\n\n'

# This section calculates the median values for the delta Psi opening angles

In [33]:
# Load .npy arrays generated in 001_nstation_binning.py 

name_mapping = {
    'Stations': 'ns',
    'Energy' : 'en',
    'Weights' : 'w',
    'Zenith' : 'zen',
    'Azimuth' : 'az',
    'Proton' : 'p',
    'Helium' : 'he',
    'Oxygen' : 'o',
    'Iron' : 'fe',
    'Laputop' : 'lap',
    'ShowerPlane' : 'sp',
    'MC' : 'mc',
    'T1' : 't1',
    'T2' : 't2',
    'T3' : 't3',
    'T4' : 't4'
}

for long_file_name in sorted(glob.glob(DIRECT+"/*")):
    file_name = long_file_name.split("/")[-1]
    parts = file_name.replace(".npy", "").split("-")
    # Map each part to its corresponding variable name component
    # Manually reorder and map parts to form the desired variable name
    reordered_parts = []
    if len(parts) == 4:
        reordered_parts.append(name_mapping.get(parts[2], parts[2].lower()))          # Always start with dataset/reco
    reordered_parts.extend([
        name_mapping.get(parts[-1], parts[-1].lower()),       # Next is 'param'
    name_mapping.get(parts[0], parts[0].lower()),  # e.g., 'Proton' -> 'p'
        name_mapping.get(parts[1], parts[1].lower()),  # e.g., 'T1' -> 't1'
    ])
    var_name = "_".join(reordered_parts)
    exec(f"{var_name} = np.load(os.path.join(DIRECT, '{file_name}'))")
    print(f"Loaded {var_name} with shape {eval(var_name).shape}")

Loaded lap_az_he_t1 with shape (32745,)
Loaded lap_w_he_t1 with shape (32745,)
Loaded lap_zen_he_t1 with shape (32745,)
Loaded mc_az_he_t1 with shape (32745,)
Loaded mc_en_he_t1 with shape (32745,)
Loaded mc_w_he_t1 with shape (32745,)
Loaded mc_zen_he_t1 with shape (32745,)
Loaded sp_az_he_t1 with shape (32745,)
Loaded sp_w_he_t1 with shape (32745,)
Loaded sp_zen_he_t1 with shape (32745,)
Loaded ns_he_t1 with shape (321584,)
Loaded lap_az_he_t2 with shape (112021,)
Loaded lap_w_he_t2 with shape (112021,)
Loaded lap_zen_he_t2 with shape (112021,)
Loaded mc_az_he_t2 with shape (112021,)
Loaded mc_en_he_t2 with shape (112021,)
Loaded mc_w_he_t2 with shape (112021,)
Loaded mc_zen_he_t2 with shape (112021,)
Loaded sp_az_he_t2 with shape (112021,)
Loaded sp_w_he_t2 with shape (112021,)
Loaded sp_zen_he_t2 with shape (112021,)
Loaded ns_he_t2 with shape (321584,)
Loaded lap_az_he_t3 with shape (85060,)
Loaded lap_w_he_t3 with shape (85060,)
Loaded lap_zen_he_t3 with shape (85060,)
Loaded mc_

In [34]:
# Concatenate several of the above arrays to compare across tiers/particles/etc.
#NOTE - NEEDS TO BE EDITED FOR 2015/2018: NO t1 and NO t2, so files will not be found!!!

# Monte Carlo 
particles = ['p', 'he', 'o', 'fe']
tiers = ['t1', 't2', 't3', 't4']
parameters = ['zen', 'az', 'w']
recos = ['mc', 'lap', 'sp']

for reco, param, tier in product(recos, parameters, tiers):
    concat_list = []
    for p in particles:
        concat_list.append(eval("_".join([reco, param, p, tier])))
    var_name = "_".join([reco, param, tier])
    exec(f"{var_name} = np.concatenate(concat_list, axis=None)")

    if reco == "mc":
        concat_list = []
        for p in particles:
            concat_list.append(eval("_".join([reco, 'en', p, tier])))
        var_name = "_".join([reco, 'en', tier])
        exec(f"{var_name} = np.concatenate(concat_list, axis=None)")

#NStations?

In [35]:

reco_name = ['ShowerPlane', 'Laputop', 'Laputop', 'Laputop']
tiers = ['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4']

# Concatenate arrays
mc_zen = [mc_zen_t1, mc_zen_t2, mc_zen_t3, mc_zen_t4]
reco_zen = [sp_zen_t1, lap_zen_t2, lap_zen_t3, lap_zen_t4]

mc_az = [mc_az_t1, mc_az_t2, mc_az_t3, mc_az_t4]
reco_az = [sp_az_t1, lap_az_t2, lap_az_t3, lap_az_t4]

# Call functions to plot
reco_open_ang_res_bytier(mc_zen, reco_zen, mc_az, reco_az, reco_name, tiers)



Median: 2.578
68%: 1.622 - 9.476
Median: 1.039
68%: 0.615 - 2.755
Median: 0.71
68%: 0.413 - 1.613
Median: 0.445
68%: 0.253 - 0.694
